#   항공편 지연 예측 머신러닝 분석
본 프로젝트는  '항공편 지연 예측 경진대회'의 100만 건 데이터를 활용하여, **데이터 전처리, 불균형 처리(SMOTE), 모델 간 비교(ROC-AUC), 하이퍼파라미터 튜닝**을 거쳐 최적의 예측 모델을 구축하는 과정을 담고 있습니다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 머신러닝 라이브러리
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_curve, auc

# 모델
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# 불균형 처리
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import joblib

# 폰트 설정 (한글 깨짐 방지 - 윈도우 맑은 고딕)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

## Step 1. 데이터 탐색 (EDA) 및 전처리

In [ ]:
print("데이터 로딩 중...")
df = pd.read_csv("dataset/train.csv")

# 정답(Delay)이 존재하는 데이터만 필터링 후 약 25만 건 전체 데이터 사용
df = df.dropna(subset=['Delay'])
# 10만건 샘플링 제한 해제 -> 약 25만건 전체 데이터 풀가동!
print(f'사용 데이터 수: {len(df)}건')

# 피처 엔지니어링 (시간 변환)
df['Hour'] = df['Estimated_Departure_Time'].fillna(1200) // 100
df['Delay'] = df['Delay'].map({'Not_Delayed': 0, 'Delayed': 1})

# 분석을 위한 수치형 데이터셋 상관관계 히트맵
num_df = df[['Month', 'Distance', 'Hour', 'Delay']].dropna()

plt.figure(figsize=(8, 6))
sns.heatmap(num_df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("수치형 변수 상관관계 히트맵")
plt.show()

## Step 2. 데이터 세트 분리 및 파이프라인 구성
- 범주형/수치형 전처리 파이프라인을 구축합니다.

In [ ]:
features = ['Month', 'Distance', 'Airline', 'Origin_Airport', 'Destination_Airport', 'Hour']
X = df[features]
y = df['Delay']

# Train-Test 분리 (8:2)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 전처리 파이프라인
num_features = ['Month', 'Distance', 'Hour']
cat_features = ['Airline', 'Origin_Airport', 'Destination_Airport']

num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])

# 전처리 적용 (SMOTE 적용을 위해 변환된 데이터 생성)
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# 파이프라인 내에서 인코딩된 피처 이름 추출 (나중에 중요도 분석에 사용)
cat_encoder = preprocessor.named_transformers_['cat'].named_steps['encoder']
cat_feature_names = cat_encoder.get_feature_names_out(cat_features)
all_feature_names = num_features + list(cat_feature_names)

## Step 3. 데이터 불균형 해결 (SMOTE)
- 압도적으로 부족한 '지연(Delayed)' 데이터를 증식시킵니다.

In [ ]:
print("SMOTE 적용 전 타겟 분포:\n", y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_processed, y_train)

print("\nSMOTE 적용 후 타겟 분포:\n", y_train_resampled.value_counts())

## Step 4. 모델 간 성능 비교 및 ROC-AUC 커브
- 로지스틱 회귀, 랜덤포레스트, XGBoost 모델을 동시에 학습하고 비교합니다.

In [ ]:
# 3개 모델 정의
models = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=50, max_depth=6, random_state=42, use_label_encoder=False, eval_metric='logloss')
}

plt.figure(figsize=(10, 8))

for name, model in models.items():
    print(f"{name} 학습 중...")
    model.fit(X_train_resampled, y_train_resampled)
    y_pred_proba = model.predict_proba(X_test_processed)[:, 1]
    
    # ROC 커브 계산
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random (AUC = 0.500)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('모델 별 ROC-AUC 커브 비교')
plt.legend(loc='lower right')
plt.show()

## Step 5. 최적 모델 튜닝 (GridSearchCV) 및 평가
- 가장 뛰어난 잠재력을 보인 **랜덤포레스트**를 선택하여 파라미터 튜닝을 진행합니다.

In [ ]:
#  지연(Delayed) 예측력(Recall/F1)을 극대화하기 위한 XGBoost 가중치(scale_pos_weight) 튜닝

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [6, 8],
    'learning_rate': [0.1]
}

# 핵심 포인트: scale_pos_weight=5 
# (정상 비행기가 지연 비행기보다 약 4.5배 많으므로, 지연 비행기를 틀렸을 때 5배의 페널티를 모델에 부여합니다!)
xgb_model = XGBClassifier(
    random_state=42, 
    use_label_encoder=False, 
    eval_metric='logloss', 
    tree_method='hist', 
    scale_pos_weight=5
)

grid_search = GridSearchCV(xgb_model, param_grid, cv=3, scoring='f1', n_jobs=-1)

print(" XGBoost (지연 잡아내기 특화 버전) 튜닝 시작...")
# SMOTE 가짜 데이터 대신, 원본 데이터에 가중치 페널티를 주는 방식이 XGBoost에서는 훨씬 효과적입니다.
grid_search.fit(X_train_processed, y_train)

print(" 최적의 파라미터:", grid_search.best_params_)
best_model = grid_search.best_estimator_

# 최종 평가
y_pred = best_model.predict(X_test_processed)
print("\n[ 튜닝 완료 XGBoost 분류 리포트 (Recall/F1 상승)]\n", classification_report(y_test, y_pred, target_names=['Not Delayed', 'Delayed']))

# 오차 행렬 시각화
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', xticklabels=['Not Delayed', 'Delayed'], yticklabels=['Not Delayed', 'Delayed'])
plt.ylabel('실제값 (Actual)')
plt.xlabel('예측값 (Predicted)')
plt.title('지연 예측 강화 모델 Confusion Matrix')
plt.show()

## Step 6. 피처 중요도 분석 및 모델 저장
- 어떤 요인이 지연에 가장 큰 영향을 미쳤는지 분석합니다.

In [ ]:
importances = best_model.feature_importances_

# 중요도 상위 10개 피처 시각화
imp_df = pd.DataFrame({'Feature': all_feature_names, 'Importance': importances})
imp_df = imp_df.sort_values(by='Importance', ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=imp_df, palette='viridis')
plt.title('상위 10개 피처 중요도 (Top 10 Feature Importances)')
plt.show()

# 최종 배포용 파이프라인 생성 (웹앱 연동용)
# 전처리기와 튜닝된 최고 모델을 합침
final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', best_model)
])

# joblib으로 저장
joblib.dump(final_pipeline, "flight_pipeline.pkl")
print(" flight_pipeline.pkl 모델 저장 완료! (웹앱에서 사용됩니다)")